In [ ]:
import json
import sys
import time
from pathlib import Path
from datetime import datetime, timezone
from collections import deque

import websocket
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# ==============================================================================
# Import Protobuf Decoder
# ==============================================================================
PROTO_PATH = Path(r"D:\Data Projects\MEXC API Architecture\websocket-proto")
sys.path.append(str(PROTO_PATH))

from PushDataV3ApiWrapper_pb2 import PushDataV3ApiWrapper

# ==============================================================================
# Configuration
# ==============================================================================
WS_URL = "wss://wbs-api.mexc.com/ws"
SYMBOL = "ETHUSDT"
INTERVAL = "Min1"

# Change this number anytime
MA_PERIOD = 7

ENDPOINT = f"spot@public.kline.v3.api.pb@{SYMBOL}@{INTERVAL}"

subscription = {
    "method": "SUBSCRIPTION",
    "params": [ENDPOINT],
    "id": 1
}

received_time = lambda: datetime.now(
    timezone.utc
).strftime("%Y-%m-%d %H:%M:%S.%f UTC")

# ==============================================================================
# Connect
# ==============================================================================
def connect():
    ws = websocket.create_connection(WS_URL)
    ws.send(json.dumps(subscription))

    print("Connected:", ENDPOINT)
    return ws

ws = connect()

# ==============================================================================
# Fixed Display
# ==============================================================================
dashboard_display = display(
    HTML(""),
    display_id=True
)

chart_display = display(
    HTML(""),
    display_id=True
)

# ==============================================================================
# History Storage
# ==============================================================================
HISTORY_SIZE = 200

open_history = deque(maxlen=HISTORY_SIZE)
high_history = deque(maxlen=HISTORY_SIZE)
low_history = deque(maxlen=HISTORY_SIZE)
close_history = deque(maxlen=HISTORY_SIZE)
volume_history = deque(maxlen=HISTORY_SIZE)

ma_history = deque(maxlen=HISTORY_SIZE)

# ==============================================================================
# Chart
# ==============================================================================
fig, ax = plt.subplots(figsize=(14,5))

close_line, = ax.plot(
    [],
    [],
    color="blue",
    linewidth=2,
    marker="o",
    markersize=3,
    label="Close"
)

ma_line, = ax.plot(
    [],
    [],
    color="green",
    linewidth=2,
    label=f"MA({MA_PERIOD})"
)

ax.set_title(f"{SYMBOL} Close Price vs MA({MA_PERIOD})")

ax.set_xlabel("Candle")

ax.set_ylabel("Price")

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02,1)
)

ax.grid(
    True,
    alpha=0.3
)

ax.set_xlim(
    0,
    HISTORY_SIZE
)

plt.tight_layout()

chart_display.update(fig)

# ==============================================================================
# Chart Update
# ==============================================================================
def update_chart():

    x = range(len(close_history))

    close_line.set_data(
        x,
        close_history
    )

    ma_values = [
        value
        for value in ma_history
        if value is not None
    ]

    ma_x = range(len(ma_values))

    ma_line.set_data(
        ma_x,
        ma_values
    )

    values = (
        list(close_history)
        +
        ma_values
    )

    if values:

        high = max(values)
        low = min(values)

        padding = (
            high - low
        ) * 0.1

        if padding == 0:
            padding = 1

        ax.set_ylim(
            low - padding,
            high + padding
        )

    ax.set_xlim(
        0,
        HISTORY_SIZE
    )

    fig.canvas.draw_idle()

# ==============================================================================
# Refresh
# ==============================================================================
DISPLAY_RATE = 1
last_update = 0

# ==============================================================================
# Receive Loop
# ==============================================================================
# Continuously receive market data from the WebSocket.
#
# If the connection is interrupted (e.g., server timeout, network issue,
# or server restart), automatically reconnect and continue receiving data.
# Existing in-memory history (charts, indicators, statistics, etc.) is
# preserved because only the WebSocket connection is recreated.
# ==============================================================================

while True:

    try:
        message = ws.recv()

    except Exception as e:
        print(
            "Connection lost:",
            e
        )

        try:
            ws.close()

        except Exception:
            pass

        time.sleep(2)
        ws = connect()

        continue

    if isinstance(message, str):

        continue

    wrapper = PushDataV3ApiWrapper()
    wrapper.ParseFromString(message)

    kline = wrapper.publicSpotKline

    # ==========================================================================
    # OHLCV
    # ==========================================================================
    window_start = datetime.fromtimestamp(
        kline.windowStart,
        tz=timezone.utc
    )

    window_end = datetime.fromtimestamp(
        kline.windowEnd,
        tz=timezone.utc
    )

    open_price = float(kline.openingPrice)

    high_price = float(kline.highestPrice)

    low_price = float(kline.lowestPrice)

    close_price = float(kline.closingPrice)

    volume = float(kline.volume)

    amount = float(kline.amount)

    # ==========================================================================
    # Store OHLCV
    # ==========================================================================
    open_history.append(open_price)

    high_history.append(high_price)

    low_history.append(low_price)

    close_history.append(close_price)

    volume_history.append(volume)

    # ==========================================================================
    # Dynamic MA Calculation
    # ==========================================================================
    ma_value = None

    if len(close_history) >= MA_PERIOD:

        ma_value = sum(
            list(close_history)[-MA_PERIOD:]
        ) / MA_PERIOD

    ma_history.append(
        ma_value
    )

    # ==========================================================================
    # Dashboard
    # ==========================================================================
    current_time = time.time()

    if current_time - last_update >= DISPLAY_RATE:

        dashboard = []

        dashboard.append("=" * 82)
        dashboard.append(f"{SYMBOL} LIVE OHLCV")
        dashboard.append("=" * 82)

        dashboard.append(f"Receive Time : {received_time()}")

        dashboard.append(f"Candle Start : {window_start}")

        dashboard.append(f"Candle End   : {window_end}")

        dashboard.append(f"Open         : {open_price:.2f}")

        dashboard.append(f"High         : {high_price:.2f}")

        dashboard.append(f"Low          : {low_price:.2f}")

        dashboard.append(f"Close        : {close_price:.2f}")

        if ma_value is not None:

            dashboard.append(f"MA({MA_PERIOD})      : {ma_value:.2f}")
        else:
            dashboard.append(f"MA({MA_PERIOD})      : N/A")

        dashboard.append(f"Volume       : {volume:.5f} ETH")

        dashboard.append(f"Amount       : {amount:,.2f} USDT")

        dashboard.append("=" * 82)

        dashboard_display.update(
            HTML(
                "<pre>"
                +
                "\n".join(dashboard)
                +
                "</pre>"
            )
        )

        update_chart()

        chart_display.update(fig)

        last_update = current_time